# Engine Model Validation Plots

Loads the newest `engine_model_validation_*.csv` from `Integrated Flight Computer/logs` and plots:
- live vs model thrust per engine
- thrust error (`live - model`) per engine
- global demand/supply/margin channels


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')

ROOT = Path.cwd()
LOG_DIR = ROOT / 'Integrated Flight Computer' / 'logs'
if not LOG_DIR.exists():
    # Fallback when notebook cwd is already inside IFC folders
    candidates = list(ROOT.rglob('engine_model_validation_*.csv'))
    if not candidates:
        raise FileNotFoundError('No engine_model_validation_*.csv found from current working directory')
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
else:
    files = sorted(LOG_DIR.glob('engine_model_validation_*.csv'), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No engine_model_validation_*.csv in {LOG_DIR}')
    latest = files[-1]

print('Latest log:', latest)
df = pd.read_csv(latest)
print('Rows:', len(df))
df.head()


In [ ]:
numeric_cols = [
    'ut_s','sample_idx','engine_idx',
    'engine_thrust_kn','engine_avail_thrust_kn','engine_maxthrust_kn',
    'model_engine_thrust_kn','thrust_error_kn','model_engine_q_demand_u_s',
    'model_spool_01','throttle_cmd_01','pilot_throttle_01',
    'mach','p_atm_atm','rho_kg_m3','q_demand_u_s','q_supply_u_s','margin_u_s','lookahead_margin_u_s',
    'worst_spool_lag_s','starving_flag','ship_intakeair_buffer_u','ship_thrust_kn','ship_avail_thrust_kn',
    'altitude_m','vertical_speed_m_s','airspeed_m_s','sound_speed_m_s','ifc_dt_s',
    'db_loaded','model_eng_count','model_intake_count'
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['t_rel_s'] = df['ut_s'] - df['ut_s'].min()
df[['t_rel_s','engine_idx','engine_name','engine_mode','engine_thrust_kn','model_engine_thrust_kn','thrust_error_kn']].head()


In [ ]:
# Global channels are repeated per engine row; take first row per sample.
global_cols = [
    'sample_idx','t_rel_s','throttle_cmd_01','pilot_throttle_01','mach','p_atm_atm','rho_kg_m3',
    'q_demand_u_s','q_supply_u_s','margin_u_s','lookahead_margin_u_s','worst_spool_lag_s',
    'starving_flag','ship_intakeair_buffer_u','ship_thrust_kn','ship_avail_thrust_kn',
    'altitude_m','vertical_speed_m_s','airspeed_m_s','sound_speed_m_s'
]
g = df.sort_values(['sample_idx','engine_idx']).groupby('sample_idx', as_index=False)[global_cols].first()

summary = {
    'duration_s': float(df['t_rel_s'].max()),
    'engines': sorted(df['engine_idx'].dropna().unique().tolist()),
    'engine_names': sorted(df['engine_name'].dropna().unique().tolist()),
    'max_live_thrust_kn': float(df['engine_thrust_kn'].max()),
    'max_model_thrust_kn': float(df['model_engine_thrust_kn'].max()),
    'mean_abs_thrust_error_kn': float(df['thrust_error_kn'].abs().mean()),
    'starving_fraction': float((g['starving_flag'] > 0).mean()),
}
summary


In [ ]:
eng_ids = sorted(df['engine_idx'].dropna().unique())
n = len(eng_ids)
fig, axes = plt.subplots(max(n,1), 2, figsize=(14, 3.2*max(n,1)), squeeze=False, sharex=False)

if n == 0:
    axes[0,0].text(0.1, 0.5, 'No engine rows found')
    axes[0,1].axis('off')
else:
    for r, eng_id in enumerate(eng_ids):
        d = df[df['engine_idx'] == eng_id].sort_values('t_rel_s')
        name = d['engine_name'].iloc[0] if not d.empty else f'engine {eng_id}'

        ax = axes[r,0]
        ax.plot(d['t_rel_s'], d['engine_thrust_kn'], label='live thrust', linewidth=1.8)
        ax.plot(d['t_rel_s'], d['model_engine_thrust_kn'], label='model thrust', linewidth=1.4)
        ax.set_title(f'Engine {eng_id} ({name}) thrust')
        ax.set_ylabel('kN')
        ax.set_xlabel('time [s]')
        ax.legend(loc='best')

        ax2 = axes[r,1]
        ax2.plot(d['t_rel_s'], d['thrust_error_kn'], color='tab:red', linewidth=1.4)
        ax2.axhline(0, color='black', linewidth=0.8)
        ax2.set_title(f'Engine {eng_id} thrust error (live - model)')
        ax2.set_ylabel('kN')
        ax2.set_xlabel('time [s]')

fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(g['t_rel_s'], g['q_demand_u_s'], label='q_demand_u_s')
axes[0].plot(g['t_rel_s'], g['q_supply_u_s'], label='q_supply_u_s')
axes[0].plot(g['t_rel_s'], g['margin_u_s'], label='margin_u_s')
axes[0].set_ylabel('u/s')
axes[0].set_title('Intake flow model channels')
axes[0].legend(loc='best')

axes[1].plot(g['t_rel_s'], g['throttle_cmd_01'], label='throttle_cmd_01')
axes[1].plot(g['t_rel_s'], g['pilot_throttle_01'], label='pilot_throttle_01', alpha=0.7)
axes[1].plot(g['t_rel_s'], g['mach'], label='mach', alpha=0.9)
axes[1].set_ylabel('unitless')
axes[1].set_title('Command / flight condition')
axes[1].legend(loc='best')

axes[2].plot(g['t_rel_s'], g['ship_thrust_kn'], label='ship_thrust_kn')
axes[2].plot(g['t_rel_s'], g['ship_avail_thrust_kn'], label='ship_avail_thrust_kn', alpha=0.8)
axes[2].set_ylabel('kN')
axes[2].set_xlabel('time [s]')
axes[2].set_title('Ship total thrust')
axes[2].legend(loc='best')

fig.tight_layout()
plt.show()
